# Example 03: VisiumHD Segmentation Statistics（增强版）

本 notebook 分析 VisiumHD 细胞与核分割的面积分布和核质比。

**Phase 0 增强图型：**
- 3-panel 分割质量图（Cell Area / Nucleus Area / NC Ratio）
- Cell vs Nucleus 联合散点图（颜色映射 NC Ratio）

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data.loaders import load_visiumhd_geojson
from src.data.validators import validate_geojson_features
from src.examples.config import VISIUMHD_DIR
from src.examples.segmentation_stats import (
    compute_areas,
    segmentation_summary,
    plot_segmentation_quality,
    plot_area_scatter,
)
from src.utils.plot_style import apply_style
apply_style()

In [ ]:
# 加载（耗时约 15-20 秒）
cell_feats = load_visiumhd_geojson(VISIUMHD_DIR, layer="cell")
nuc_feats = load_visiumhd_geojson(VISIUMHD_DIR, layer="nucleus")
print(f"Cell features: {len(cell_feats)}")
print(f"Nucleus features: {len(nuc_feats)}")
print(f"Cell 校验: {validate_geojson_features(cell_feats) or 'OK'}")
print(f"Nucleus 校验: {validate_geojson_features(nuc_feats) or 'OK'}")

In [ ]:
# 计算面积
cell_areas = compute_areas(cell_feats)
nuc_areas = compute_areas(nuc_feats)

# 汇总
stats = segmentation_summary(cell_areas, nuc_areas)
for k, v in stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# 面积分布可视化
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cell_areas["area"], bins=80, color="steelblue", edgecolor="white")
axes[0].set_title("Cell Area Distribution")
axes[0].set_xlabel("Area (px^2)")
axes[1].hist(nuc_areas["area"], bins=80, color="salmon", edgecolor="white")
axes[1].set_title("Nucleus Area Distribution")
axes[1].set_xlabel("Area (px^2)")
fig.suptitle("VisiumHD Segmentation Statistics")
fig.tight_layout()
plt.show()

## Phase 0 增强可视化

In [ ]:
# 增强 3-panel 分割质量图（含 NC Ratio）
import matplotlib.pyplot as plt
fig = plot_segmentation_quality(cell_areas, nuc_areas)
plt.show()

In [ ]:
# Cell vs Nucleus 联合散点图
fig = plot_area_scatter(cell_areas, nuc_areas)
plt.show()